In [ ]:
!pip install --upgrade pip setuptools wheel

In [ ]:
!pip install -q datasets transformers seqeval

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "parquet",
    data_files={
        "train": "hf://datasets/ai4bharat/naamapadam@~parquet/hi/train/0000.parquet",
        "validation": "hf://datasets/ai4bharat/naamapadam@~parquet/hi/validation/0000.parquet",
        "test": "hf://datasets/ai4bharat/naamapadam@~parquet/hi/test/0000.parquet"
    }
)

print(dataset)
print(dataset["train"].features)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 931000
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 13460
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 867
    })
})
{'tokens': List(Value('string')), 'ner_tags': List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']))}


In [ ]:
print(dataset["train"][0])

{'tokens': ['सेक्टर', '55/56', 'के', 'एसएचओ', 'अरविंद', 'कुमार', 'ने', 'बताया', 'कि', 'इस', 'मामले', 'में', 'आईपीसी', 'की', 'धारा', '376', '-', 'डी', '(', 'गैंगरेप', ')', 'के', 'तहत', 'मामला', 'दर्ज', 'कर', 'लिया', 'गया', 'है', '।'], 'ner_tags': [0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [ ]:
label_names = dataset["train"].features["ner_tags"].feature.names

for i, label in enumerate(label_names):
    print(i, label)

0 O
1 B-PER
2 I-PER
3 B-ORG
4 I-ORG
5 B-LOC
6 I-LOC


In [ ]:
def find_bad_rows(ds):
    bad = []
    for i, example in enumerate(ds):
        if len(example["tokens"]) != len(example["ner_tags"]):
            bad.append({
                "index": i,
                "tokens": len(example["tokens"]),
                "tags": len(example["ner_tags"])
            })
    return bad

bad_train = find_bad_rows(dataset["train"])
print("Bad rows:", len(bad_train))
print(bad_train[:10])

Bad rows: 0
[]


In [ ]:
import unicodedata

def normalize_tokens(example):
    example["tokens"] = [
        unicodedata.normalize("NFC", token)
        for token in example["tokens"]
    ]
    return example

dataset = dataset.map(
    normalize_tokens,
    desc="NFC normalization"
)

for split in dataset:
    print(split, len(dataset[split]))

train 931000
validation 13460
test 867


In [ ]:
import numpy as np

lengths = [
    len(x["tokens"])
    for x in dataset["train"]
]

print("Mean:", np.mean(lengths))
print("Median:", np.median(lengths))
print("Max:", np.max(lengths))

Mean: 22.34987647690655
Median: 20.0
Max: 100


In [ ]:
from collections import Counter

sample = dataset["train"].shuffle(seed=42).select(range(50_000))
counts = Counter(t for tags in sample["ner_tags"] for t in tags)

total = sum(counts.values())
for i, name in enumerate(label_names):
    print(f"{name:8s} {counts.get(i,0):>9,}  {counts.get(i,0)/total*100:5.2f}%")

O          914,650  82.21%
B-PER       38,855   3.49%
I-PER       35,861   3.22%
B-ORG       34,608   3.11%
I-ORG       41,564   3.74%
B-LOC       36,786   3.31%
I-LOC       10,292   0.93%


In [ ]:
def extract_spans(tags):
    spans, start = [], None
    orphan_i = 0
    for i, t in enumerate(tags):
        name = label_names[t]
        if name.startswith("B-"):
            if start is not None:
                spans.append((label_names[tags[start]][2:], i - start))
            start = i
        elif name.startswith("I-"):
            if start is None:
                start = i
                orphan_i += 1
        else:
            if start is not None:
                spans.append((label_names[tags[start]][2:], i - start))
                start = None
    if start is not None:
        spans.append((label_names[tags[start]][2:], len(tags) - start))
    return spans, orphan_i

spans_per_doc, span_lens, type_counts = [], [], Counter()
total_orphans = 0
for tags in sample["ner_tags"]:
    sp, orphans = extract_spans(tags)
    total_orphans += orphans
    spans_per_doc.append(len(sp))
    for typ, ln in sp:
        span_lens.append(ln)
        type_counts[typ] += 1

print("Spans/doc  mean:", np.mean(spans_per_doc), " median:", np.median(spans_per_doc))
print("Span length mean:", np.mean(span_lens), " max:", np.max(span_lens))
print("Entity type counts:", dict(type_counts))
print("Orphan I-tags (silver-label noise):", total_orphans)

Spans/doc  mean: 2.20498  median: 2.0
Span length mean: 1.7956262641838021  max: 57
Entity type counts: {'PER': 38855, 'LOC': 36786, 'ORG': 34608}
Orphan I-tags (silver-label noise): 0


In [ ]:
train_100k = dataset["train"].shuffle(seed=42).select(range(100_000))
val = dataset["validation"]
test = dataset["test"]
print(len(train_100k), len(val), len(test))

100000 13460 867


In [ ]:
c100k = Counter(t for tags in train_100k.select(range(10_000))["ner_tags"] for t in tags)
tot = sum(c100k.values())
for i, name in enumerate(label_names):
    print(f"{name:8s} {c100k.get(i,0)/tot*100:5.2f}%")

O        82.43%
B-PER     3.45%
I-PER     3.18%
B-ORG     3.04%
I-ORG     3.69%
B-LOC     3.33%
I-LOC     0.90%


In [ ]:
from datasets import DatasetDict

processed = DatasetDict({"train": train_100k, "validation": val, "test": test})
processed.save_to_disk("/content/naamapadam_hi_100k")

Saving the dataset (0/1 shards):   0%|          | 0/100000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/13460 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/867 [00:00<?, ? examples/s]

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p "/content/drive/MyDrive/indic-ner"
!cp -r /content/naamapadam_hi_100k "/content/drive/MyDrive/indic-ner/naamapadam_hi_100k"
print("Saved to Drive: MyDrive/indic-ner/naamapadam_hi_100k")

Mounted at /content/drive
Saved to Drive: MyDrive/indic-ner/naamapadam_hi_100k
